This notebook compares the GPT-2 from Hugging Face with the implementation from TransformerLens.


In [1]:
import torch
import torch.nn.functional as F
from transformer_lens import HookedTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model_hf = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer_hf = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
model_tl = HookedTransformer.from_pretrained(
    "gpt2-small",
    center_unembed=False,
    center_writing_weights=False,
    fold_ln=False,
    refactor_factored_attn_matrices=False,
    device="cpu",
)

model_tl_simple = HookedTransformer.from_pretrained(
    "gpt2-small",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=True,
    device="cpu",
)

tokenizer_tl = model_tl.tokenizer

In [4]:
prompt = "GPT2 is a model developed by OpenAI."
input_ids = tokenizer_hf(prompt, return_tensors="pt").input_ids

In [ ]:
tokenizer_hf(prompt).input_ids

In [ ]:
tokenizer_tl(prompt).input_ids

In [7]:
logits_hf = model_hf(input_ids).logits
logits_tl = model_tl(input_ids)

In [ ]:
torch.isclose(logits_hf, logits_tl).all()

In [9]:
logits_tl_simple = model_tl_simple(input_ids)

In [ ]:
torch.isclose(
    F.softmax(logits_hf, dim=-1), F.softmax(logits_tl_simple, dim=-1), atol=2e-5, rtol=0
).all()

In [11]:
hf_output = model_hf(input_ids, output_hidden_states=True)
hidden_states_hf = hf_output.hidden_states

In [ ]:
for i, hs in enumerate(hidden_states_hf):
    print(i, hs.shape)

In [ ]:
model_tl

In [14]:
tl_output = model_tl.run_with_cache(input_ids)

In [ ]:
for key in tl_output[1].keys():
    print(key, tl_output[1][key].shape)

In [ ]:
for i, hidden_state in enumerate(hidden_states_hf):
    found_match = False

    for hook in tl_output[1].keys():
        ht_acts = tl_output[1][hook]

        if hidden_state.shape == ht_acts.shape:
            diff = (ht_acts - hidden_state).abs().max()
            if diff < 0.001:
                found_match = True
                print(f"hidden state {i} = {hook}\t({diff})")

    if not found_match:
        print(f"hidden state {i} = no match")